In [ ]:
import pandas as pd
from pysradb.sraweb import SRAweb
from tqdm import tqdm
import time
import socket

# 1. 초기 설정
with open("./SRA_ids.txt") as f:
    srr_list = [line.strip() for line in f if line.strip()]

db = SRAweb()
all_results = []
chunk_size = 50 

def fetch_with_retry(ids, max_retries=5):
    for attempt in range(max_retries):
        try:
            df = db.sra_metadata(ids, detailed=True)
            return df
        except (Exception, socket.gaierror) as e:
            wait_time = (attempt + 1) * 5 
            if attempt < max_retries - 1:
                print(f"\n[Network Error] {e}. Retrying in {wait_time}s... ({attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"\n[Final Failure] Could not fetch data for IDs starting with {ids[0]}")
                return None

for i in tqdm(range(0, len(srr_list), chunk_size), desc="1st Fetching"):
    chunk = srr_list[i:i + chunk_size]
    df_chunk = fetch_with_retry(chunk)
    
    if df_chunk is not None and not df_chunk.empty:
        all_results.append(df_chunk)
    
    time.sleep(1) 

/home/mjcho/miniconda3/envs/scRNA/lib/python3.10/site-packages/pysradb/utils.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
1st Fetching:  21%|██        | 15/73 [01:44<06:13,  6.45s/it]


[Network Error] HTTPSConnectionPool(host='eutils.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /entrez/eutils/esearch.fcgi (Caused by NameResolutionError("HTTPSConnection(host='eutils.ncbi.nlm.nih.gov', port=443): Failed to resolve 'eutils.ncbi.nlm.nih.gov' ([Errno -3] Temporary failure in name resolution)")). Retrying in 5s... (1/5)


1st Fetching: 100%|██████████| 73/73 [09:00<00:00,  7.41s/it]


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [2]:
def make_columns_unique(df):
    if df is None or df.empty:
        return df
    
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        dup_indices = cols[cols == dup].index
        for i, idx in enumerate(dup_indices):
            if i > 0:
                cols[idx] = f"{dup}_{i}"
    df.columns = cols
    return df

all_results_uniq = []
for df_chunk in all_results :
    all_results_uniq.append(make_columns_unique(df_chunk))
final_df = pd.concat(all_results_uniq, axis=0, ignore_index=True, sort=False) if all_results else pd.DataFrame()
collected_ids = set(final_df['run_accession'].unique()) if not final_df.empty else set()
missing_ids = list(set(srr_list) - collected_ids)

if missing_ids:
    print(f"\n[Recovery] Attempting to recover {len(missing_ids)} missing samples individually...")
    for m_id in tqdm(missing_ids, desc="2nd Recovery"):
        df_m = fetch_with_retry([m_id], max_retries=3)
        if df_m is not None and not df_m.empty:
            final_df = pd.concat([final_df, df_m], axis=0, ignore_index=True)
        time.sleep(0.5)

final_df = final_df.drop_duplicates(subset=['run_accession'])
author = pd.read_table("./meta_author.tsv", index_col=0)
author = author[~author.index.duplicated(keep='first')]
meta_with_author = final_df.merge(author, left_on='bioproject', right_on='BioProject', how='left')

print(f"\nTarget: {len(srr_list)}, Final: {len(meta_with_author)}")


Target: 3634, Final: 3634


In [5]:
meta_with_author.to_csv("./meta_raw.tsv", sep='\t', index=False)